# TA OOF 보정 + HM CatBoost/residual LSTM

TA 실험 2의 rolling OOF 예측을 최근 OOF 연도 잔차로 보정한 뒤, HM CatBoost와 12~14시 residual LSTM을 학습합니다. HM 보정층의 TA 후보는 실제 ASOS TA가 아니라 OOF/frozen 모델 예측만 사용합니다. 마지막 셀에서 `RMSE_TA + 0.1 × RMSE_HM`을 출력합니다.

In [ ]:
# 1. Drive 안전 마운트: /content/drive가 비어 있지 않으면 별도 경로 사용
from pathlib import Path
from google.colab import drive
import os

MOUNT_ROOT = Path('/content/drive')
if not os.path.ismount(MOUNT_ROOT):
    if MOUNT_ROOT.exists() and any(MOUNT_ROOT.iterdir()):
        MOUNT_ROOT = Path('/content/gdrive')
    drive.mount(str(MOUNT_ROOT), force_remount=False)
MYDRIVE = MOUNT_ROOT / 'MyDrive'
print('MyDrive:', MYDRIVE)

In [ ]:
# 2. 전용 GitHub 브랜치 동기화
import subprocess, shutil

REPO_URL = 'https://github.com/tswaincae1221/SME_DATA.git'
REPO_BRANCH = 'agent/ta-hm-oof-calibrated'
REPO_DIR = Path('/content/SME_DATA_ta_hm_oof')
if (REPO_DIR / '.git').exists():
    subprocess.run(['git', 'fetch', 'origin', REPO_BRANCH], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], cwd=REPO_DIR, check=True)
else:
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)
print('repo:', REPO_DIR)

In [ ]:
# 3. Colab 의존성
%pip install -q catboost==1.2.8
import torch, catboost, pandas as pd
print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'catboost', catboost.__version__)

In [ ]:
# 4. 입력/출력 경로 — Drive 폴더가 다르면 이 셀만 수정
DATA_ROOT = MYDRIVE / 'SME_DATA' / 'processed_station_features'
MASTER_CSV = DATA_ROOT / 'final_train_dataset_19to25_master.csv'
SHORTTERM_CSV = DATA_ROOT / 'shortterm_12to14_data' / 'incremental_12to14_tables' / 'shortterm_long_2019to2025.csv'
TA_OUTPUT = DATA_ROOT / 'model_experiments_19to25' / 'ta_exp123_rolling_residual'
OUTPUT_DIR = DATA_ROOT / 'model_experiments_19to25' / 'ta_hm_oof_calibrated_baseline'
TA_OOF_CSV = TA_OUTPUT / 'rolling_oof_scored_predictions.csv'
TA_TEST_CSV = TA_OUTPUT / 'test_2025_predictions.csv'

for label, path in [('master', MASTER_CSV), ('shortterm', SHORTTERM_CSV)]:
    if not path.exists():
        raise FileNotFoundError(f'{label} 파일이 없습니다: {path}')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('master:', MASTER_CSV)
print('shortterm:', SHORTTERM_CSV)
print('output:', OUTPUT_DIR)

In [ ]:
# 5. 앞선 TA rolling OOF 산출물이 없을 때만 자동 재생성
import sys

if not (TA_OOF_CSV.exists() and TA_TEST_CSV.exists()):
    TA_OUTPUT.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, '-u', str(REPO_DIR / 'scripts' / 'experiment_ta_calibration_residual_ablation.py'),
        '--master-csv', str(MASTER_CSV),
        '--shortterm-long-csv', str(SHORTTERM_CSV),
        '--output-dir', str(TA_OUTPUT),
        '--device', 'auto', '--threads', '4',
    ]
    print('$', ' '.join(cmd))
    subprocess.run(cmd, cwd=REPO_DIR, check=True)
else:
    print('기존 TA OOF 산출물을 사용합니다:', TA_OUTPUT)

In [ ]:
# 6. TA OOF 보정 + HM 베이스라인 학습/평가
cmd = [
    sys.executable, '-u', str(REPO_DIR / 'scripts' / 'train_ta_hm_oof_calibrated_baseline.py'),
    '--master-csv', str(MASTER_CSV),
    '--shortterm-long-csv', str(SHORTTERM_CSV),
    '--ta-oof-csv', str(TA_OOF_CSV),
    '--ta-test-csv', str(TA_TEST_CSV),
    '--output-dir', str(OUTPUT_DIR),
    '--device', 'auto', '--threads', '4',
]
print('$', ' '.join(cmd))
subprocess.run(cmd, cwd=REPO_DIR, check=True)

In [ ]:
# 7. 대회식 점수와 선택 결과 확인
import json
from IPython.display import display

scores = pd.read_csv(OUTPUT_DIR / 'competition_scores.csv')
ta_metrics = pd.read_csv(OUTPUT_DIR / 'ta_calibration_metrics.csv')
hm_metrics = pd.read_csv(OUTPUT_DIR / 'hm_metrics.csv')
display(scores)
display(ta_metrics[ta_metrics['split'].str.startswith('test_')])
display(hm_metrics[hm_metrics['split'].eq('test_2025')])
summary = json.loads((OUTPUT_DIR / 'experiment_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary['final_test_metrics'], ensure_ascii=False, indent=2))
print('모든 결과와 모델이 Drive에 저장되었습니다:', OUTPUT_DIR)